In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

In [ ]:
USE_AMP = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("AMP Enabled:", USE_AMP and device.type == "cuda")

scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

Using device: cuda
AMP Enabled: True


/tmp/ipykernel_55/70618383.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))


Dataset Loader

In [ ]:
# ===============================
# Dataset Loader (70-10-20 SPLIT ON TRAIN SET ONLY)
# ===============================
def get_dataloaders(dataset_name, batch_size, pm):

    transform = transforms.Compose([
        transforms.Resize(224),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ])

    # Load ONLY the official training dataset
    if dataset_name == "MNIST":
        full_dataset = datasets.MNIST(
            "./data", train=True, download=True, transform=transform
        )

    elif dataset_name == "FashionMNIST":
        full_dataset = datasets.FashionMNIST(
            "./data", train=True, download=True, transform=transform
        )

    total_len = len(full_dataset)

    train_len = int(0.7 * total_len)
    val_len   = int(0.1 * total_len)
    test_len  = total_len - train_len - val_len

    train_set, val_set, test_set = torch.utils.data.random_split(
        full_dataset,
        [train_len, val_len, test_len],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True, pin_memory=pm
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False, pin_memory=pm
    )
    test_loader = DataLoader(
        test_set, batch_size=batch_size, shuffle=False, pin_memory=pm
    )

    return train_loader, val_loader, test_loader

Model Loader

In [ ]:
def get_model(model_name):
    if model_name == "resnet18":
        model = models.resnet18(pretrained=False)
    elif model_name == "resnet50":
        model = models.resnet50(pretrained=False)

    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

Optimizer

In [ ]:
def get_optimizer(optimizer_name, model, lr):
    if optimizer_name == "SGD":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optimizer_name == "Adam":
        return optim.Adam(model.parameters(), lr=lr)

Model Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

In [ ]:
def validation_loss(model, loader, criterion):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item()

    return running_loss / len(loader)

Accuracy Evaluation

In [ ]:
def test_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                outputs = model(images)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100.0 * correct / total

In [ ]:
datasets_list   = ["MNIST", "FashionMNIST"]
models_list     = ["resnet18", "resnet50"]
batch_sizes     = [16, 32]
optimizers_list = ["SGD", "Adam"]
learning_rates  = [0.001, 0.0001]
epochs_list     = [3, 2]
pms             = [True, False]

results = []

for epochs in epochs_list:
    for pm in pms:
        for dataset_name in datasets_list:
            for model_name in models_list:
                for batch_size in batch_sizes:
                    for optimizer_name in optimizers_list:
                        for lr in learning_rates:

                            print(f"\nDataset={dataset_name}, Model={model_name}, "
                                  f"BS={batch_size}, Opt={optimizer_name}, LR={lr}, "
                                  f"Epochs={epochs}, pin_memory={pm},  AMP={USE_AMP}")

                            train_loader, val_loader, test_loader = get_dataloaders(
                                dataset_name, batch_size, pm
                            )

                            model = get_model(model_name)
                            optimizer = get_optimizer(optimizer_name, model, lr)
                            criterion = nn.CrossEntropyLoss()

                            for epoch in range(epochs):
                                train_loss = train_one_epoch(
                                    model, train_loader, optimizer, criterion
                                )
                                val_loss = validation_loss(
                                    model, val_loader, criterion
                                )
                                print(
                                    f"Epoch [{epoch+1}/{epochs}] "
                                    f"Train Loss: {train_loss:.4f} | "
                                    f"Val Loss: {val_loss:.4f}"
                                )

                            #val_acc = test_accuracy(model, val_loader)
                            test_acc = test_accuracy(model, test_loader)

                            #print(f"Validation Accuracy: {val_acc:.2f}%")
                            print(f"Test Accuracy: {test_acc:.2f}%")

                            results.append([
                                dataset_name, model_name, batch_size,
                                optimizer_name, lr, epochs, pm, test_acc
                            ])


Dataset=MNIST, Model=resnet18, BS=16, Opt=SGD, LR=0.001, Epochs=3, pin_memory=True,  AMP=True


100%|██████████| 9.91M/9.91M [00:00<00:00, 12.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 338kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.19MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.36MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/tmp/ipykernel_55/1167330303.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
/tmp/ipykernel_55/1311880033.py:10: FutureWa

Epoch [1/3] Train Loss: 0.3456 | Val Loss: 0.0613
Epoch [2/3] Train Loss: 0.0649 | Val Loss: 0.0400
Epoch [3/3] Train Loss: 0.0425 | Val Loss: 0.0334
Test Accuracy: 98.94%

Dataset=MNIST, Model=resnet18, BS=16, Opt=SGD, LR=0.0001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 1.4155 | Val Loss: 0.6555
Epoch [2/3] Train Loss: 0.4060 | Val Loss: 0.2015
Epoch [3/3] Train Loss: 0.2111 | Val Loss: 0.1492
Test Accuracy: 96.15%

Dataset=MNIST, Model=resnet18, BS=16, Opt=Adam, LR=0.001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 0.1355 | Val Loss: 0.0698
Epoch [2/3] Train Loss: 0.0615 | Val Loss: 0.0639
Epoch [3/3] Train Loss: 0.0480 | Val Loss: 0.0540
Test Accuracy: 98.27%

Dataset=MNIST, Model=resnet18, BS=16, Opt=Adam, LR=0.0001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 0.1439 | Val Loss: 0.0565
Epoch [2/3] Train Loss: 0.0489 | Val Loss: 0.0314
Epoch [3/3] Train Loss: 0.0375 | Val Loss: 0.0236
Test Accuracy: 99.33%

Dataset=MNIST, Model=r

100%|██████████| 26.4M/26.4M [00:00<00:00, 109MB/s] 
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.69MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 59.6MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 12.1MB/s]


Epoch [1/3] Train Loss: 0.6098 | Val Loss: 0.4136
Epoch [2/3] Train Loss: 0.3442 | Val Loss: 0.2877
Epoch [3/3] Train Loss: 0.2777 | Val Loss: 0.2586
Test Accuracy: 90.48%

Dataset=FashionMNIST, Model=resnet18, BS=16, Opt=SGD, LR=0.0001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 1.2015 | Val Loss: 0.7537
Epoch [2/3] Train Loss: 0.6699 | Val Loss: 0.5591
Epoch [3/3] Train Loss: 0.5256 | Val Loss: 0.8349
Test Accuracy: 70.98%

Dataset=FashionMNIST, Model=resnet18, BS=16, Opt=Adam, LR=0.001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 0.4991 | Val Loss: 0.4547
Epoch [2/3] Train Loss: 0.3165 | Val Loss: 0.3244
Epoch [3/3] Train Loss: 0.2635 | Val Loss: 0.2509
Test Accuracy: 90.91%

Dataset=FashionMNIST, Model=resnet18, BS=16, Opt=Adam, LR=0.0001, Epochs=3, pin_memory=True,  AMP=True
Epoch [1/3] Train Loss: 0.4419 | Val Loss: 0.3063
Epoch [2/3] Train Loss: 0.2714 | Val Loss: 0.2483
Epoch [3/3] Train Loss: 0.2201 | Val Loss: 0.2809
Test Accuracy: 90.47%

D

KeyboardInterrupt: 

In [ ]:
import torch

# Path where the model will be saved
MODEL_PATH = "Q1_a.pth"

# Save only model parameters (recommended)
torch.save(model.state_dict(), MODEL_PATH)

print(f"Model saved to {MODEL_PATH}")


NameError: name 'model' is not defined